[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/learning_methods/01_linear_regression/Linear_Regression.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/learning_methods/01_linear_regression/Linear_Regression.ipynb)

# Notebook 01 — Linear Regression
## Predicting a Number from Data

**Dataset**: Titanic passengers
**Question we answer**: *How much did a passenger's ticket cost?*
**Learning type**: Supervised Regression — the answer is a continuous number

---

## Section 1 — What Is Linear Regression?

### In plain English

Imagine you work at a travel agency and must estimate how much a Titanic ticket would cost.
You notice patterns in the data:

- First-class passengers paid more than third-class
- Larger families tended to pay more (group bookings)
- Passengers boarding at Cherbourg paid differently than Southampton

**Linear regression finds the best formula to combine all these clues into one predicted number:**

```
predicted_fare = (w1 × pclass) + (w2 × sex) + (w3 × age) + ... + bias
```

Each feature gets a **weight** — how much it influences the prediction.

| Weight | Meaning |
|---|---|
| Large positive | Higher feature value → higher fare |
| Large negative | Higher feature value → lower fare |
| Near zero | Feature barely matters |

The formula draws a straight-line relationship through the data — that is why it is called *linear*.

## Section 2 — How Does It Learn?

### Three steps, repeated thousands of times

**Step 1 — Predict**
Multiply each feature by its current weight and sum them up. First time: weights are random, prediction is terrible.

**Step 2 — Measure error**
Compare predicted fare to actual fare using Mean Squared Error:
```
MSE = average of (predicted − actual)²
```
Squaring penalises large errors more than small ones.

**Step 3 — Update weights (Gradient Descent)**
Compute which direction each weight must move to reduce error.
Take a tiny step in that direction. Repeat.

> **Analogy**: Walking downhill in thick fog. You cannot see the valley but you feel the slope.
> Take a small step downhill. Check the slope again. Repeat until flat.
> The slope is the *gradient*. The valley is minimum error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

weights = np.linspace(-3, 5, 300)
error   = (weights - 1.5)**2 + 0.3

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(weights, error, 'steelblue', lw=2.5, label='Error surface')
ax.set_xlabel('Weight value', fontsize=12)
ax.set_ylabel('Error (MSE)', fontsize=12)
ax.set_title('Gradient Descent: Walking Downhill to Find the Best Weight', fontsize=12)

steps = [4.2, 3.3, 2.5, 2.0, 1.7, 1.52]
for i in range(len(steps) - 1):
    y0 = (steps[i]   - 1.5)**2 + 0.3
    y1 = (steps[i+1] - 1.5)**2 + 0.3
    ax.annotate('', xy=(steps[i+1], y1), xytext=(steps[i], y0),
                arrowprops=dict(arrowstyle='->', color='orangered', lw=2))
ax.scatter(steps, [(s-1.5)**2+0.3 for s in steps], color='orangered', zorder=5, s=60)
ax.axvline(1.5, color='green', ls='--', lw=1.5, label='Best weight (minimum error)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()
print('Each red arrow = one gradient descent step.')
print('Steps get smaller as we approach the minimum — this is convergence.')

## Section 3 — What Does the Data Need?

### Why scaling is essential for gradient descent

Gradient descent adjusts weights based on the raw size of feature values.
- `age` ranges 0–80
- `sex_enc` ranges 0–1

The `age` feature pushes the gradient 80× harder than `sex_enc`.
Result: some weights overshoot, others barely move — learning is slow and unstable.

**Fix: StandardScaler** — transforms every feature to mean=0 and std=1.
Now all features push the gradient with equal force.

### Per-feature prep table

| Feature | Raw range | Distribution | Transform | Why |
|---|---|---|---|---|
| `pclass` | 1, 2, 3 | Ordinal | StandardScaler | Uncentred range |
| `sex_enc` | 0, 1 | Binary | StandardScaler | Applied uniformly |
| `age` | 0–80 | ≈ Normal | StandardScaler | Must centre |
| `sibsp` | 0–8 | Zero-heavy count | StandardScaler | Sparse range |
| `parch` | 0–6 | Zero-heavy count | StandardScaler | Sparse range |
| `embarked_enc` | 0, 1, 2 | Ordinal | StandardScaler | Applied uniformly |
| `family_size` | 1–11 | Right-skewed | StandardScaler | Wide range |

> `fare` is the **target** here — we predict it, we do not use it as a feature.

### The split rule

Always split train/test **first**, then fit the scaler on training data only.
Fitting on all data before splitting leaks test statistics into the scaler — **data leakage**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

df_raw = sns.load_dataset('titanic')
print(f"Titanic dataset: {df_raw.shape[0]} passengers, {df_raw.shape[1]} columns")
df_raw[['survived','pclass','sex','age','sibsp','parch','fare','embarked']].head(6)

In [ ]:
df = df_raw.copy()
df['age']          = df['age'].fillna(df['age'].median())
df['fare']         = df['fare'].fillna(df['fare'].median())
df['embarked']     = df['embarked'].fillna(df['embarked'].mode()[0])
df['sex_enc']      = (df['sex'] == 'female').astype(int)
df['embarked_enc'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df['family_size']  = df['sibsp'] + df['parch'] + 1
df = df.dropna(subset=['survived'])
print(f"Clean dataset: {len(df)} passengers")
df[['pclass','sex_enc','age','sibsp','parch','embarked_enc','family_size','fare']].head(6)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

FEATURES = ['pclass','sex_enc','age','sibsp','parch','embarked_enc','family_size']
TARGET   = 'fare'

X_raw = df[FEATURES].values
y     = df[TARGET].values

X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_raw)
X_test  = scaler.transform(X_te_raw)

stats = pd.DataFrame({
    'Feature'     : FEATURES,
    'Raw mean'    : X_tr_raw.mean(axis=0).round(2),
    'Raw std'     : X_tr_raw.std(axis=0).round(2),
    'Scaled mean' : X_train.mean(axis=0).round(4),
    'Scaled std'  : X_train.std(axis=0).round(4),
})
print("After StandardScaler — training set statistics:")
print(stats.to_string(index=False))

## Section 4 — Train the Model and Read the Rules

Once trained, the **weights ARE the rules**.

- Positive weight on `pclass`? Higher class number → higher fare (or lower, read the sign)
- Large |weight| → strong influence on fare prediction
- Weight near 0 → feature barely matters

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
r2     = r2_score(y_test, y_pred)

print(f"RMSE : {rmse:.2f}  (predictions off by ~£{rmse:.0f} on average)")
print(f"R²   : {r2:.3f}  ({r2*100:.1f}% of fare variation explained by features)")
print()

coef_df = pd.DataFrame({'Feature': FEATURES, 'Weight': model.coef_.round(3)})
coef_df = coef_df.reindex(coef_df['Weight'].abs().sort_values(ascending=False).index)
print(f"Intercept (base fare): £{model.intercept_:.2f}")
print()
print("Feature weights (sorted by magnitude):")
print(coef_df.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(y_test, y_pred, alpha=0.3, s=18, color='steelblue')
lim = max(y_test.max(), y_pred.max()) * 1.05
ax1.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect prediction')
ax1.set_xlabel('Actual fare (£)', fontsize=11)
ax1.set_ylabel('Predicted fare (£)', fontsize=11)
ax1.set_title(f'Actual vs Predicted Fare\nRMSE={rmse:.1f}  R²={r2:.3f}', fontsize=12)
ax1.legend()

colors = ['#26A69A' if w >= 0 else '#EF5350' for w in coef_df['Weight']]
ax2.barh(coef_df['Feature'], coef_df['Weight'], color=colors)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_title('Feature Weights\n(green = raises fare  red = lowers fare)', fontsize=12)
ax2.set_xlabel('Weight', fontsize=11)

plt.tight_layout()
plt.show()